# 21. 组合优化器

## 学习目标

通过本次学习，你将能够：

1. **整合 CAPM + 因子模型 + 优化技能**
2. **构建完整的组合优化工具**
3. **理解选股逻辑和约束条件的业务含义**
4. **输出权重饼图 + 有效前沿 + 与等权对比**

## 知识地图

```
组合优化器
├── 选股逻辑
│   ├── 基于 CAPM 的 Beta 筛选
│   ├── 基于因子模型的因子暴露
│   └── 多条件综合筛选
├── 有效前沿
│   ├── 均值-方差优化
│   ├── 最大夏普比组合
│   └── 最小方差组合
├── 约束条件
│   ├── 权重上下限
│   ├── 行业分散化
│   └── 换手率限制
├── 回溯测试
│   ├── 样本内表现
│   ├── 样本外表现
│   └── 与基准对比
└── 可视化
    ├── 权重饼图
    ├── 有效前沿图
    └── 累积收益对比
```

## 环境依赖

```bash
pip install numpy pandas matplotlib scipy
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 设置随机种子
np.random.seed(42)

%matplotlib inline

---
## 1. 理论基础：组合优化

### 1.1 什么是组合优化？

组合优化是**在给定风险水平下最大化收益**，或**在给定收益水平下最小化风险**的过程。

核心公式（Markowitz 均值-方差模型）：

$$\min_w \frac{1}{2} w^T \Sigma w$$

$$\text{s.t.} \quad w^T \mu = \mu_p, \quad w^T \mathbf{1} = 1, \quad w_i \geq 0$$

其中：
- $w$：权重向量
- $\Sigma$：协方差矩阵
- $\mu$：预期收益率向量
- $\mu_p$：目标收益率

### 1.2 有效前沿

有效前沿是所有**风险-收益最优组合**的集合：

- 对于每个风险水平，有效前沿上的组合提供最高收益
- 对于每个收益水平，有效前沿上的组合提供最低风险
- 理性投资者只会选择有效前沿上的组合

### 1.3 关键组合

| 组合 | 特点 | 适用场景 |
|------|------|----------|
| **最小方差组合** | 风险最低 | 风险厌恶型投资者 |
| **最大夏普比组合** | 风险调整后收益最高 | 追求性价比的投资者 |
| **等权组合** | 每只股票权重相等 | 简单分散化 |
| **市值加权组合** | 按市值分配权重 | 被动指数投资 |

---
## 2. 数据准备：选股逻辑

### 2.1 选股标准

我们基于以下标准选择 5-10 只股票：

1. **市值要求**：选择大中盘股票（市值 > 中位数）
2. **盈利能力**：ROE > 行业平均
3. **估值合理**：PE < 行业平均
4. **流动性**：日均成交额 > 1000 万

### 2.2 模拟数据生成

In [ ]:
def generate_stock_data(n_stocks=20, n_periods=252*2):
    """
    生成模拟股票数据
    n_stocks: 股票数量
    n_periods: 交易日数量（2年）
    """
    np.random.seed(42)
    
    # 股票名称
    stock_names = [
        '贵州茅台', '中国平安', '招商银行', '宁德时代', '比亚迪',
        '美的集团', '恒瑞医药', '海康威视', '万华化学', '隆基绿能',
        '长江电力', '中国神华', '紫金矿业', '药明康德', '迈瑞医疗',
        '东方财富', '中信证券', '三一重工', '中联重科', '中国中免'
    ][:n_stocks]
    
    # 行业分类
    industries = [
        '白酒', '金融', '金融', '新能源', '新能源',
        '家电', '医药', '安防', '化工', '新能源',
        '电力', '煤炭', '有色金属', '医药', '医药',
        '金融', '金融', '机械', '机械', '零售'
    ][:n_stocks]
    
    # 股票特征
    market_caps = np.exp(np.random.normal(12, 1, n_stocks))  # 市值（亿元）
    pe_ratios = np.random.uniform(10, 50, n_stocks)  # PE
    roe = np.random.uniform(0.05, 0.25, n_stocks)  # ROE
    daily_volume = np.random.uniform(5000, 50000, n_stocks)  # 日均成交额（万元）
    
    # 生成收益率（带行业相关性）
    dates = pd.date_range('2022-01-01', periods=n_periods, freq='B')
    
    # 市场收益
    market_returns = np.random.normal(0.0003, 0.015, n_periods)
    
    # 行业因子
    unique_industries = list(set(industries))
    industry_returns = {}
    for ind in unique_industries:
        industry_returns[ind] = np.random.normal(0.0002, 0.01, n_periods)
    
    # 生成个股收益
    returns_data = {}
    for i, stock in enumerate(stock_names):
        beta = 0.5 + 1.0 * np.random.random()  # Beta: 0.5-1.5
        alpha = 0.0001 * np.random.randn()  # Alpha
        
        # 个股收益 = alpha + beta*市场 + 行业因子 + 个股噪声
        stock_returns = (
            alpha 
            + beta * market_returns 
            + 0.5 * industry_returns[industries[i]] 
            + np.random.normal(0, 0.02, n_periods)
        )
        
        returns_data[stock] = stock_returns
    
    # 构建 DataFrame
    returns_df = pd.DataFrame(returns_data, index=dates)
    
    # 股票信息
    stock_info = pd.DataFrame({
        'stock': stock_names,
        'industry': industries,
        'market_cap': market_caps,
        'pe_ratio': pe_ratios,
        'roe': roe,
        'daily_volume': daily_volume
    })
    
    return returns_df, stock_info, market_returns


# 生成数据
returns_df, stock_info, market_returns = generate_stock_data()

print("股票信息:")
print(stock_info.to_string(index=False))
print(f"\n收益率数据维度: {returns_df.shape}")
print(f"时间范围: {returns_df.index[0]} 到 {returns_df.index[-1]}")

### 2.3 基于条件筛选股票

In [ ]:
def screen_stocks(stock_info, returns_df, n_select=10):
    """
    多条件股票筛选
    """
    # 复制数据避免修改原始数据
    df = stock_info.copy()
    
    # 条件1: 市值 > 中位数（大中盘）
    median_cap = df['market_cap'].median()
    df = df[df['market_cap'] > median_cap]
    
    # 条件2: ROE > 平均值（盈利能力强）
    mean_roe = df['roe'].mean()
    df = df[df['roe'] > mean_roe]
    
    # 条件3: PE < 平均值（估值合理）
    mean_pe = df['pe_ratio'].mean()
    df = df[df['pe_ratio'] < mean_pe]
    
    # 条件4: 日均成交额 > 1000 万（流动性好）
    df = df[df['daily_volume'] > 1000]
    
    # 计算综合得分
    df['score'] = (
        0.3 * (df['roe'] - df['roe'].min()) / (df['roe'].max() - df['roe'].min()) +
        0.3 * (1 - (df['pe_ratio'] - df['pe_ratio'].min()) / (df['pe_ratio'].max() - df['pe_ratio'].min())) +
        0.2 * (df['market_cap'] - df['market_cap'].min()) / (df['market_cap'].max() - df['market_cap'].min()) +
        0.2 * (df['daily_volume'] - df['daily_volume'].min()) / (df['daily_volume'].max() - df['daily_volume'].min())
    )
    
    # 按得分排序，选择前 n_select 只
    df = df.sort_values('score', ascending=False).head(n_select)
    
    return df


# 筛选股票
selected_stocks = screen_stocks(stock_info, returns_df, n_select=10)
print(f"筛选出 {len(selected_stocks)} 只股票：\n")
print(selected_stocks[['stock', 'industry', 'market_cap', 'pe_ratio', 'roe', 'score']].to_string(index=False))

# 获取选中股票的收益率数据
selected_returns = returns_df[selected_stocks['stock'].tolist()]
print(f"\n选中股票收益率数据维度: {selected_returns.shape}")

---
## 3. 有效前沿构建

### 3.1 计算预期收益率和协方差矩阵

In [ ]:
# 计算年化收益率和协方差矩阵
annual_returns = selected_returns.mean() * 252  # 年化收益率
cov_matrix = selected_returns.cov() * 252  # 年化协方差矩阵

print("预期年化收益率：")
for stock, ret in annual_returns.items():
    print(f"  {stock}: {ret:.2%}")

print(f"\n协方差矩阵形状: {cov_matrix.shape}")

# 相关系数矩阵
corr_matrix = selected_returns.corr()

# 可视化相关系数矩阵
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix, cmap='RdYlBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_matrix)))
ax.set_yticks(range(len(corr_matrix)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(corr_matrix.columns, fontsize=9)
ax.set_title('选中股票的相关系数矩阵', fontsize=14)

# 添加数值标签
for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix)):
        ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                ha='center', va='center', fontsize=8,
                color='white' if abs(corr_matrix.iloc[i, j]) > 0.5 else 'black')

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

### 3.2 优化函数定义

In [ ]:
def portfolio_stats(weights, returns, cov_matrix):
    """
    计算组合的收益率、波动率和夏普比
    """
    port_return = np.dot(weights, returns)
    port_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    sharpe_ratio = port_return / port_volatility  # 假设无风险利率为0
    return port_return, port_volatility, sharpe_ratio


def min_variance(returns, cov_matrix, target_return=None):
    """
    最小方差组合
    """
    n = len(returns)
    init_weights = np.array([1/n] * n)
    
    # 约束条件
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]  # 权重和为1
    
    if target_return is not None:
        constraints.append({'type': 'eq', 'fun': lambda w: np.dot(w, returns) - target_return})
    
    # 边界条件（不允许做空）
    bounds = tuple((0, 0.3) for _ in range(n))  # 每只股票最多30%
    
    # 优化目标：最小化方差
    def objective(w):
        return np.dot(w.T, np.dot(cov_matrix, w))
    
    result = minimize(objective, init_weights, method='SLSQP',
                      bounds=bounds, constraints=constraints)
    
    return result.x


def max_sharpe(returns, cov_matrix):
    """
    最大夏普比组合
    """
    n = len(returns)
    init_weights = np.array([1/n] * n)
    
    # 约束条件
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
    
    # 边界条件
    bounds = tuple((0, 0.3) for _ in range(n))
    
    # 优化目标：最大化夏普比（最小化负夏普比）
    def objective(w):
        port_return = np.dot(w, returns)
        port_volatility = np.sqrt(np.dot(w.T, np.dot(cov_matrix, w)))
        return -port_return / port_volatility
    
    result = minimize(objective, init_weights, method='SLSQP',
                      bounds=bounds, constraints=constraints)
    
    return result.x


# 计算不同组合
n_stocks = len(selected_stocks)

# 等权组合
equal_weights = np.array([1/n_stocks] * n_stocks)

# 最小方差组合
min_var_weights = min_variance(annual_returns.values, cov_matrix.values)

# 最大夏普比组合
max_sharpe_weights = max_sharpe(annual_returns.values, cov_matrix.values)

# 计算各组合的统计量
portfolios = {
    '等权组合': equal_weights,
    '最小方差组合': min_var_weights,
    '最大夏普比组合': max_sharpe_weights
}

print("组合对比：")
print(f"{'组合':<15} {'收益率':>10} {'波动率':>10} {'夏普比':>10}")
print("-" * 50)

for name, weights in portfolios.items():
    ret, vol, sharpe = portfolio_stats(weights, annual_returns.values, cov_matrix.values)
    print(f"{name:<15} {ret:>10.2%} {vol:>10.2%} {sharpe:>10.2f}")

### 3.3 绘制有效前沿

In [ ]:
# 生成有效前沿
target_returns = np.linspace(annual_returns.min(), annual_returns.max(), 50)
frontier_volatilities = []
frontier_returns = []

for target in target_returns:
    try:
        weights = min_variance(annual_returns.values, cov_matrix.values, target_return=target)
        ret, vol, _ = portfolio_stats(weights, annual_returns.values, cov_matrix.values)
        frontier_returns.append(ret)
        frontier_volatilities.append(vol)
    except:
        pass

# 绘制有效前沿
fig, ax = plt.subplots(figsize=(12, 8))

# 有效前沿
ax.plot(frontier_volatilities, frontier_returns, 'b-', linewidth=3, label='有效前沿')

# 标记各组合
colors = {'等权组合': 'red', '最小方差组合': 'green', '最大夏普比组合': 'purple'}
markers = {'等权组合': 'o', '最小方差组合': 's', '最大夏普比组合': '^'}

for name, weights in portfolios.items():
    ret, vol, sharpe = portfolio_stats(weights, annual_returns.values, cov_matrix.values)
    ax.scatter(vol, ret, s=200, c=colors[name], marker=markers[name], 
               label=f'{name}\n(σ={vol:.1%}, r={ret:.1%}, SR={sharpe:.2f})', zorder=5)

# 标记个股
for i, stock in enumerate(selected_stocks['stock']):
    stock_return = annual_returns[stock]
    stock_vol = np.sqrt(cov_matrix.iloc[i, i])
    ax.scatter(stock_vol, stock_return, s=50, alpha=0.5, color='gray')
    ax.annotate(stock, (stock_vol, stock_return), fontsize=8, alpha=0.7)

ax.set_xlabel('波动率 (年化)', fontsize=12)
ax.set_ylabel('收益率 (年化)', fontsize=12)
ax.set_title('有效前沿与投资组合', fontsize=14)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

# 设置坐标轴范围
ax.set_xlim(0.10, 0.30)
ax.set_ylim(0.05, 0.25)

plt.tight_layout()
plt.show()

print("\n解读：")
print("  - 有效前沿上的每个点都代表一个最优组合")
print("  - 最小方差组合在有效前沿的最左端（风险最低）")
print("  - 最大夏普比组合在有效前沿的切点位置（风险调整后收益最高）")
print("  - 等权组合通常不在有效前沿上（不是最优的）")

---
## 4. 组合权重分析

### 4.1 权重饼图

In [ ]:
# 绘制权重饼图
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

stock_names = selected_stocks['stock'].tolist()

for idx, (name, weights) in enumerate(portfolios.items()):
    ax = axes[idx]
    
    # 过滤掉权重很小的股票
    threshold = 0.02
    significant = weights > threshold
    plot_weights = weights[significant]
    plot_names = [stock_names[i] for i in range(len(stock_names)) if significant[i]]
    
    # 加上"其他"
    other_weight = 1 - plot_weights.sum()
    if other_weight > 0:
        plot_weights = np.append(plot_weights, other_weight)
        plot_names.append('其他')
    
    # 绘制饼图
    colors = plt.cm.Set3(np.linspace(0, 1, len(plot_weights)))
    wedges, texts, autotexts = ax.pie(
        plot_weights, 
        labels=plot_names, 
        autopct='%1.1f%%',
        colors=colors,
        pctdistance=0.85,
        startangle=90
    )
    
    # 设置字体大小
    for text in texts:
        text.set_fontsize(9)
    for autotext in autotexts:
        autotext.set_fontsize(8)
    
    ax.set_title(name, fontsize=12)

plt.suptitle('不同组合的权重分配', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# 打印详细权重
print("\n详细权重分配：")
print(f"{'股票':<10} {'等权':>10} {'最小方差':>10} {'最大夏普':>10}")
print("-" * 45)
for i, stock in enumerate(stock_names):
    print(f"{stock:<10} {equal_weights[i]:>10.2%} {min_var_weights[i]:>10.2%} {max_sharpe_weights[i]:>10.2%}")

### 4.2 权重解读

In [ ]:
# 分析各组合的特征
print("组合特征分析：\n")

for name, weights in portfolios.items():
    print(f"=== {name} ===")
    
    # 计算组合的加权 PE、ROE
    weighted_pe = np.dot(weights, selected_stocks['pe_ratio'].values)
    weighted_roe = np.dot(weights, selected_stocks['roe'].values)
    weighted_cap = np.dot(weights, selected_stocks['market_cap'].values)
    
    # 计算行业分布
    industry_weights = {}
    for i, stock in selected_stocks.iterrows():
        ind = stock['industry']
        if ind not in industry_weights:
            industry_weights[ind] = 0
        industry_weights[ind] += weights[selected_stocks.index.get_loc(i)]
    
    print(f"  加权 PE: {weighted_pe:.1f}")
    print(f"  加权 ROE: {weighted_roe:.1%}")
    print(f"  加权市值: {weighted_cap:.0f} 亿")
    print(f"  行业分布:")
    for ind, w in sorted(industry_weights.items(), key=lambda x: -x[1]):
        if w > 0.01:
            print(f"    {ind}: {w:.1%}")
    print()

---
## 5. 回溯测试

### 5.1 样本内回溯

In [ ]:
def backtest_portfolio(returns_df, weights, rebalance_freq='M'):
    """
    回溯测试组合表现
    """
    # 计算组合收益
    portfolio_returns = (returns_df * weights).sum(axis=1)
    
    # 计算累积收益
    cumulative_returns = (1 + portfolio_returns).cumprod()
    
    # 计算回撤
    peak = cumulative_returns.expanding().max()
    drawdown = (cumulative_returns - peak) / peak
    
    return {
        'returns': portfolio_returns,
        'cumulative': cumulative_returns,
        'drawdown': drawdown,
        'total_return': cumulative_returns.iloc[-1] - 1,
        'annual_return': (cumulative_returns.iloc[-1] ** (252/len(returns_df))) - 1,
        'annual_volatility': portfolio_returns.std() * np.sqrt(252),
        'sharpe_ratio': (portfolio_returns.mean() * 252) / (portfolio_returns.std() * np.sqrt(252)),
        'max_drawdown': drawdown.min()
    }


# 回溯测试各组合
backtest_results = {}
for name, weights in portfolios.items():
    backtest_results[name] = backtest_portfolio(selected_returns, weights)

# 打印回溯结果
print("回溯测试结果（样本内）：\n")
print(f"{'组合':<15} {'总收益':>10} {'年化收益':>10} {'年化波动':>10} {'夏普比':>10} {'最大回撤':>10}")
print("-" * 70)

for name, result in backtest_results.items():
    print(f"{name:<15} {result['total_return']:>10.2%} {result['annual_return']:>10.2%} "
          f"{result['annual_volatility']:>10.2%} {result['sharpe_ratio']:>10.2f} {result['max_drawdown']:>10.2%}")

### 5.2 累积收益对比

In [ ]:
# 绘制累积收益对比
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# 累积收益
ax1 = axes[0]
colors = {'等权组合': 'red', '最小方差组合': 'green', '最大夏普比组合': 'purple'}

for name, result in backtest_results.items():
    ax1.plot(result['cumulative'].index, result['cumulative'], 
             label=name, color=colors[name], linewidth=2)

ax1.set_xlabel('日期', fontsize=12)
ax1.set_ylabel('累积收益', fontsize=12)
ax1.set_title('组合累积收益对比', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# 回撤
ax2 = axes[1]
for name, result in backtest_results.items():
    ax2.fill_between(result['drawdown'].index, result['drawdown'], 0,
                     alpha=0.3, color=colors[name], label=name)

ax2.set_xlabel('日期', fontsize=12)
ax2.set_ylabel('回撤', fontsize=12)
ax2.set_title('组合回撤对比', fontsize=14)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

plt.tight_layout()
plt.show()

print("\n解读：")
print("  - 最大夏普比组合通常有最高的风险调整后收益")
print("  - 最小方差组合的回撤通常最小")
print("  - 等权组合的表现介于两者之间")
print("  - 样本内的优异表现不一定能延续到样本外")

---
## 6. 约束条件的业务含义

### 6.1 为什么需要约束？

| 约束 | 数学形式 | 业务含义 |
|------|----------|----------|
| **权重非负** | $w_i \geq 0$ | 不允许做空（多数公募基金限制） |
| **权重上限** | $w_i \leq 30\%$ | 避免过度集中于单只股票 |
| **权重和为1** | $\sum w_i = 1$ | 全额投资，不持有现金 |
| **行业分散** | $\sum_{i \in \text{industry}} w_i \leq 40\%$ | 避免行业集中风险 |

### 6.2 约束对优化结果的影响

In [ ]:
# 对比有无约束的结果
def min_variance_unconstrained(returns, cov_matrix):
    """
    无约束的最小方差组合（允许做空）
    """
    n = len(returns)
    init_weights = np.array([1/n] * n)
    
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
    bounds = tuple((-0.5, 1.0) for _ in range(n))  # 允许做空到-50%
    
    def objective(w):
        return np.dot(w.T, np.dot(cov_matrix, w))
    
    result = minimize(objective, init_weights, method='SLSQP',
                      bounds=bounds, constraints=constraints)
    return result.x


# 计算无约束组合
unconstrained_weights = min_variance_unconstrained(annual_returns.values, cov_matrix.values)
unconstrained_ret, unconstrained_vol, unconstrained_sharpe = portfolio_stats(
    unconstrained_weights, annual_returns.values, cov_matrix.values)

# 对比
print("约束条件对优化结果的影响：\n")
print(f"{'组合':<20} {'收益率':>10} {'波动率':>10} {'夏普比':>10}")
print("-" * 55)

for name, weights in portfolios.items():
    ret, vol, sharpe = portfolio_stats(weights, annual_returns.values, cov_matrix.values)
    print(f"{name:<20} {ret:>10.2%} {vol:>10.2%} {sharpe:>10.2f}")

print(f"{'无约束最小方差':<20} {unconstrained_ret:>10.2%} {unconstrained_vol:>10.2%} {unconstrained_sharpe:>10.2f}")

print(f"\n解读：")
print(f"  - 无约束组合可能包含做空头寸")
print(f"  - 约束条件会降低优化效率，但更符合实际投资限制")
print(f"  - 权重上限30%是为了避免过度集中风险")

---
## 7. 小结

### 核心收获

1. **组合优化**是在风险和收益之间寻找最优平衡

2. **有效前沿**是所有最优组合的集合：
   - 最小方差组合：风险最低
   - 最大夏普比组合：风险调整后收益最高

3. **选股逻辑**应该有明确的经济含义：
   - 市值要求：大中盘股票更稳定
   - 盈利能力：ROE 高的公司更优质
   - 估值合理：PE 低的股票更便宜

4. **约束条件**是实际投资的必要限制：
   - 权重非负：不允许做空
   - 权重上限：避免集中风险
   - 行业分散：避免行业风险

5. **回溯测试**是验证策略的必要步骤：
   - 样本内表现不代表样本外表现
   - 要关注夏普比和最大回撤

### 关键公式

- 组合收益：$R_p = \sum w_i R_i$
- 组合方差：$\sigma_p^2 = w^T \Sigma w$
- 夏普比：$SR = (R_p - R_f) / \sigma_p$
- 有效前沿：$\min \sigma_p^2$ s.t. $R_p = \mu_p$

### 验收标准 Checklist

- [x] **能解释选股逻辑**：基于市值、ROE、PE 的多条件筛选
- [x] **约束条件有业务含义**：权重非负、上限、行业分散
- [x] **优化结果有业务解读**：有效前沿、最大夏普比组合
- [x] **上传 GitHub**：完整的组合优化工具

---

**恭喜你完成了组合优化器的学习！** 🎉

你现在已经掌握了从选股到组合优化的完整流程，可以构建自己的量化投资组合了。